# Tahap C proyek: analisis data eksploratif (EDA)

> **Data sintetis.** Temuan di sini tentang dataset buatan, bukan tentang postingan nyata
> (lihat `../DATA_CARD.md`). Teks yang ditampilkan seluruhnya sintetis.

Prasyarat: `pip install -r requirements-ml.txt`. Jalankan sel dari atas ke bawah.

In [1]:
import sys
import warnings
from pathlib import Path

# Notebook ini berada di examples/<folder-level>/. Paket nlp_hub ada di examples/.
sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore", message="'count' is passed as positional argument")

## 1. Membaca dan memeriksa struktur

In [2]:
import pandas as pd
from nlp_hub.data import muat_dataset

df = muat_dataset(hapus_duplikat=False)
print(df.shape)
print(df.dtypes)
print("nilai hilang:", df.isna().sum().sum(), "| duplikat teks:", int(df["text"].duplicated().sum()))

(574, 6)
source_id      str
event_id       str
post_type      str
template_id    str
text           str
label          str
dtype: object
nilai hilang: 0 | duplikat teks: 10


## 2. Distribusi label dan jenis postingan
Label pada data ini ditentukan oleh jenis templat, jadi tabel silang di bawah memang sempurna; pada data nyata tidak akan demikian.

In [3]:
print(df["label"].value_counts())
pd.crosstab(df["post_type"], df["label"])

label
Non-Emergency    373
Emergency        201
Name: count, dtype: int64


label,Emergency,Non-Emergency
post_type,,
bantuan,78,0
berita,0,49
candaan,0,37
edukasi,0,43
info_lama,0,40
laporan,123,0
opini,0,43
peringatan,0,39
tidak_terkait,0,122


## 3. Panjang teks
Perhatikan apakah panjang teks berbeda menurut label: itu bisa menjadi jalan pintas (*shortcut*) bagi model.

In [4]:
df = df.assign(panjang=df["text"].str.len(), n_kata=df["text"].str.split().str.len())
df.groupby("label")[["panjang", "n_kata"]].agg(["mean", "std", "min", "max"]).round(1)

panjang                n_kata             
                 mean   std min  max   mean  std min max
label                                                   
Emergency        96.7  16.5  63  148   14.8  2.1  10  21
Non-Emergency    81.1  20.8  40  145   11.1  2.3   6  18

## 4. Memeriksa kemungkinan kebocoran

In [5]:
from nlp_hub.data import bagi_latih_uji

latih, uji = bagi_latih_uji(muat_dataset())
print("event_id beririsan:", len(set(latih["event_id"]) & set(uji["event_id"])))
print("templat uji yang juga ada di latih:", len(set(uji["template_id"]) & set(latih["template_id"])), "dari", uji["template_id"].nunique())

event_id beririsan: 0
templat uji yang juga ada di latih: 59 dari 61


**Membaca hasil:** pembagian per kejadian tidak menyisakan irisan `event_id`, tetapi hampir semua templat uji juga muncul di
latih. Skor tinggi pada uji sintetis karena itu tidak boleh dibaca sebagai kemampuan menggeneralisasi.

## Latihan
1. Tambahkan histogram panjang teks per label (matplotlib) dan tulis satu kalimat interpretasi.
2. Hitung kata yang paling sering pada tiap label setelah `praproses`. Mana yang tampak seperti artefak templat?